In [ ]:
import numpy as np
import scipy.special
import matplotlib.colors
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.visualization
import named_arrays as na
import optika

In [ ]:
radius_aperture = 100 * u.mm

In [ ]:
f_number = 5

So the focal length of the primary is then

In [ ]:
focal_length = 2 * f_number * radius_aperture
focal_length

In [ ]:
sag_primary = optika.sags.ParabolicSag(-focal_length)

In [ ]:
material_primary = optika.materials.Mirror()

In [ ]:
aperture_primary = optika.apertures.CircularAperture(radius_aperture)

In [ ]:
translation_primary = na.transformations.Cartesian3dTranslation(z=focal_length)

In [ ]:
primary = optika.surfaces.Surface(
    name="primary",
    sag=sag_primary,
    material=material_primary,
    aperture=aperture_primary,
    transformation=translation_primary,
    is_pupil_stop=True,
)

In [ ]:
width_pixel = 15 * u.um

In [ ]:
num_pixel = na.Cartesian2dVectorArray(256, 256)

In [ ]:
axis_pixel = na.Cartesian2dVectorArray("detector_x", "detector_y")

In [ ]:
rotation_sensor = na.transformations.Cartesian3dRotationY(180 * u.deg)

In [ ]:
sensor = optika.sensors.ImagingSensor(
    name="sensor",
    width_pixel=width_pixel,
    axis_pixel=axis_pixel,
    num_pixel=num_pixel,
    transformation=rotation_sensor,
    is_field_stop=True,
)

In [ ]:
pupil = na.Cartesian2dVectorLinearSpace(
    start=-1,
    stop=1,
    axis=na.Cartesian2dVectorArray("px", "py"),
    num=5,
    centers=True,
)

In [ ]:
field = na.Cartesian2dVectorLinearSpace(
    start=-1,
    stop=1,
    axis=na.Cartesian2dVectorArray("fx", "fy"),
    num=5,
    centers=True,
)

In [ ]:
wavelength = 500 * u.nm

In [ ]:
grid_input = optika.vectors.ObjectVectorArray(
    wavelength=wavelength,
    field=field,
    pupil=pupil,
)

In [ ]:
system = optika.systems.SequentialSystem(
    surfaces=[
        primary,
    ],
    sensor=sensor,
    grid_input=grid_input,
)

In [ ]:
# plot the system
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    ax.set_aspect("equal")
    system.plot(
        ax=ax,
        components=("z", "y"),
        kwargs_rays=dict(
            color="tab:blue",
        ),
        color="black",
        zorder=10,
    )

In [ ]:
fig, ax = system.spot_diagram()

In [ ]:
field_psf = na.Cartesian2dVectorArray(
    x=na.ScalarArray(np.array([0, 0.45, 0.9]), axes="field_psf"),
    y=0,
)
label_field = na.ScalarArray(
    np.array([f"normalized field $x$ = {x}" for x in field_psf.x.ndarray]),
    axes="field_psf",
)

In [ ]:
axis_psf = ("psf_x", "psf_y")
psf = system.psf(
    wavelength=wavelength,
    field=field_psf,
    width_sensor=60 * u.um,
    num_sensor=121,
    axis=axis_psf,
)

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = na.plt.subplots(
        axis_cols="field_psf",
        ncols=field_psf.x.shape["field_psf"],
        figsize=(9, 3.5),
        sharey=True,
        constrained_layout=True,
    )
    mappable = na.plt.pcolormesh(
        psf.inputs.position.to(u.um),
        C=psf.outputs / psf.outputs.max(axis_psf),
        ax=axs,
        norm=matplotlib.colors.LogNorm(vmin=1e-4, vmax=1),
    )
    na.plt.set_aspect("equal", ax=axs)
    na.plt.set_xlabel("$x$ (μm)", ax=axs)
    na.plt.set_ylabel("$y$ (μm)", ax=axs[{"field_psf": 0}])
    na.plt.set_title(label_field, ax=axs)
    fig.colorbar(
        mappable=mappable[{"field_psf": 0}].ndarray,
        ax=axs.ndarray,
        label="intensity relative to the peak",
    )

In [ ]:
psf_center = psf[{"field_psf": 0}]
cut = psf_center[{axis_psf[1]: psf_center.shape[axis_psf[1]] // 2}]
x = cut.inputs.position.x

x_airy = na.linspace(-30, 30, axis="x_airy", num=1000) * u.um
argument = np.pi * x_airy / (wavelength * f_number)
argument = argument.to(u.dimensionless_unscaled).value
airy = np.square(2 * scipy.special.j1(argument) / argument)

with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    na.plt.plot(
        x.to(u.um),
        cut.outputs / cut.outputs.max(),
        ax=ax,
        axis=axis_psf[0],
        label="psf()",
    )
    na.plt.plot(
        x_airy,
        airy,
        ax=ax,
        axis="x_airy",
        linestyle="--",
        label="Airy pattern",
    )
    ax.set_yscale("log")
    ax.set_ylim(1e-5, 2)
    ax.set_xlabel("$x$ (μm)")
    ax.set_ylabel("intensity relative to the peak")
    ax.legend()

In [ ]:
radius_ee = optika.wavefields.encircled_energy_radius(
    intensity=psf.outputs,
    position=psf.inputs.position,
    axis=axis_psf,
    fraction=0.8,
)
radius_ee.to(u.um)

In [ ]:
mtf = optika.wavefields.mtf(
    intensity=psf.outputs,
    position=psf.inputs.position,
    axis=axis_psf,
)
mtf = mtf[{axis_psf[1]: mtf.shape[axis_psf[1]] // 2}]
frequency = mtf.inputs.x.to(1 / u.mm)

frequency_cutoff = (1 / (wavelength * f_number)).to(1 / u.mm)
ratio = na.linspace(0, 1, axis="frequency_limit", num=101)
mtf_limit = 2 / np.pi * (np.arccos(ratio) - ratio * np.sqrt(1 - np.square(ratio)))

with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    na.plt.plot(
        frequency,
        mtf.outputs,
        ax=ax,
        axis=axis_psf[0],
        label=label_field,
    )
    na.plt.plot(
        ratio * frequency_cutoff,
        mtf_limit,
        ax=ax,
        axis="frequency_limit",
        color="black",
        linestyle="--",
        label="diffraction limit",
    )
    ax.set_xlim(0, 1.1 * frequency_cutoff)
    ax.set_xlabel(f"spatial frequency along $x$ ({ax.get_xlabel()})")
    ax.set_ylabel("MTF")
    ax.legend()